In [6]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [1]:
!pip install -q transformers accelerate bitsandbytes torch torchvision pillow openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.7 MB/s eta 0:00:00


In [2]:
import os
import re
import glob
import json
import gc
import random
from pathlib import Path
from PIL import Image
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from openai import OpenAI

In [ ]:
IMAGE_DIR = "/content/gdrive/MyDrive/3DReasoningProject_images/Others"

# Output root (will create per-pair subfolders)
OUTPUT_ROOT = "two_view_pipeline_results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# LLaVA model and LLM model names (change if you use different models)
LLAVA_MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
INSTRUCTION_LLM_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct:novita"

# Resize images before sending to LLaVA (reduce if you hit memory errors)
IMAGE_SIZE = (512, 512)  # (width, height)

# Number of tasks per level to request from the instruction LLM
NUM_PER_LEVEL = 3

# Environment variable name for HF router API key
HF_API_KEY = os.environ.get("HF_API_KEY", "YOUR_HF_API_KEY_HERE")

# Allowed image extensions (case-insensitive)
ALLOWED_EXTS = {".png", ".jpg", ".jpeg"}

In [8]:
def find_image_pairs(image_dir):
  files = sorted(glob.glob(os.path.join(image_dir, "*")))
  pairs = {}
  # Regex to capture base and suffix _1/_2 before extension
  r1 = re.compile(r"^(.+)_1\.(png|jpg|jpeg)$", flags=re.IGNORECASE)
  r2 = re.compile(r"^(.+)_2\.(png|jpg|jpeg)$", flags=re.IGNORECASE)
  for f in files:
      name = os.path.basename(f)
      m1 = r1.match(name)
      m2 = r2.match(name)
      if m1:
          base = m1.group(1)
          pairs.setdefault(base, {})["1"] = f
      elif m2:
          base = m2.group(1)
          pairs.setdefault(base, {})["2"] = f
  # keep only bases with both '1' and '2'
  pairs_complete = {b: v for b, v in pairs.items() if "1" in v and "2" in v}
  return pairs_complete

In [9]:
find_image_pairs(IMAGE_DIR)

{'34_Pedestrian mall with.4': {'1': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/34_Pedestrian mall with.4_1.png',
  '2': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/34_Pedestrian mall with.4_2.png'},
 '35_Bus loop with.3': {'1': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.3_1.png',
  '2': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.3_2.png'},
 '35_Bus loop with.4': {'1': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.4_1.png',
  '2': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.4_2.png'},
 '37_Fortress yard with.4': {'1': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/37_Fortress yard with.4_1.png',
  '2': '/content/gdrive/MyDrive/3DReasoningProject_images/Others/37_Fortress yard with.4_2.png'}}

In [10]:
class SceneUnderstanding:
  def __init__(self, model_name=LLAVA_MODEL_NAME, image_size=IMAGE_SIZE, quantize=True):
      self.device = "cuda" if torch.cuda.is_available() else "cpu"
      print(f"Loading LLaVA model '{model_name}' on {self.device} ...")
      quantization_config = None
      if quantize:
          quantization_config = BitsAndBytesConfig(
              load_in_4bit=True,
              bnb_4bit_compute_dtype=torch.float16,
              bnb_4bit_use_double_quant=True,
              bnb_4bit_quant_type="nf4"
          )
      self.processor = AutoProcessor.from_pretrained(model_name)
      self.model = LlavaForConditionalGeneration.from_pretrained(
          model_name,
          quantization_config=quantization_config,
          device_map="auto",
          low_cpu_mem_usage=True
      )
      self.image_size = image_size
      print("✓ LLaVA ready")

  def _resize_image(self, path):
      im = Image.open(path).convert("RGB")
      im = im.resize(self.image_size, Image.BICUBIC)
      return im

  def _extract_object_list(self, text):
      if not text:
          return []
      m = re.search(r"OBJECT_LIST\s*:\s*(.+)$", text, flags=re.IGNORECASE | re.MULTILINE)
      if not m:
          return []
      raw = m.group(1).strip()
      raw = raw.strip("[]()")
      parts = [p.strip() for p in raw.split(",")]
      objs = []
      for p in parts:
          p = re.sub(r"[^a-zA-Z0-9 _-]+", "", p).strip()
          if len(p) >= 2:
              objs.append(p)
      return objs[:50]

  def analyze_two_views(self, topdown_path, frontal_path):
      assert os.path.exists(topdown_path), f"Top-down image not found: {topdown_path}"
      assert os.path.exists(frontal_path), f"Frontal image not found: {frontal_path}"

      im_top = self._resize_image(topdown_path)
      im_front = self._resize_image(frontal_path)
      images = [im_top, im_front]

      prompt = (
          "USER: <image>\n <image>\n"
          "You are given TWO images of the SAME SCENE: one TOP-DOWN / bird's-eye view and one FRONTAL view. "
          "Treat these images as multiple views of the same environment. Do NOT describe each image separately. "
          "Produce ONE unified SCENE DESCRIPTION that includes:\n"
          "  1) concise overall scene description (room/area type, key objects)\n"
          "  2) layout and navigation-relevant notes (blocked vs free areas)\n"
          "  3) relative positions of major objects if helpful\n\n"
          "IMPORTANT: At the VERY END output exactly one line that begins with:\n"
          "OBJECT_LIST: obj1, obj2, obj3, ...\n\n"
          "ASSISTANT:"
      )

      inputs = self.processor(text=prompt, images=images, return_tensors="pt").to(self.device)
      with torch.no_grad():
          output_ids = self.model.generate(**inputs, max_new_tokens=512, do_sample=False)
      raw_text = self.processor.decode(output_ids[0], skip_special_tokens=True)
      if "ASSISTANT:" in raw_text:
          raw_text = raw_text.split("ASSISTANT:")[-1].strip()

      valid_objects = self._extract_object_list(raw_text)
      description_clean = re.sub(r"\n?OBJECT_LIST\s*:\s*.+$", "", raw_text, flags=re.IGNORECASE | re.MULTILINE).strip()

      unified = "ENVIRONMENT ANALYSIS\n\n" + description_clean + "\n\nVALID_OBJECTS (auto-extracted):\n"
      unified += ", ".join(valid_objects) if valid_objects else "unknown"
      unified += "\n"

      return {
          "raw_output": raw_text,
          "clean_description": description_clean,
          "valid_objects": valid_objects,
          "unified": unified
      }

In [11]:
class InstructionGenerator:
    def __init__(self, model_name=INSTRUCTION_LLM_MODEL, hf_api_key=HF_API_KEY):
        if not hf_api_key:
            print("WARNING: HF_API_KEY not found. Instruction calls may fail.")
        self.client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_api_key)
        self.model_name = model_name
        self.BANNED_LOCATION_TERMS = {
            "left", "right", "front", "back", "center", "centre", "middle",
            "left side", "right side", "front side", "back side",
            "top", "bottom", "upper", "lower"
        }

    def _norm(self, s):
        s = (s or "").lower().strip()
        s = re.sub(r"[^a-z0-9 _-]+", " ", s)
        s = re.sub(r"\s+", " ", s).strip()
        return s

    def _contains_banned(self, s):
        t = self._norm(s)
        for term in self.BANNED_LOCATION_TERMS:
            if term in t:
                return True
        return False

    def _contains_any_object(self, text, valid_objects):
        t = self._norm(text)
        for v in valid_objects:
            vn = self._norm(v)
            if vn and vn in t:
                return True
        return False

    def _pick_two_objects(self, valid_objects):
        if not valid_objects:
            return None, None
        if len(valid_objects) == 1:
            return valid_objects[0], valid_objects[0]
        a, b = random.sample(valid_objects, 2)
        return a, b

    def _filter_interact_list(self, interact_str, valid_objects):
        if not interact_str or interact_str.lower().strip() == "none":
            return "none"
        valid = {self._norm(v) for v in valid_objects if v}
        items = [self._norm(x) for x in interact_str.split(",")]
        kept = []
        for it in items:
            if not it:
                continue
            if it in valid:
                kept.append(it)
                continue
            for v in valid:
                if v and (v in it or it in v):
                    kept.append(v)
                    break
        kept = list(dict.fromkeys(kept))
        return ", ".join(kept) if kept else "none"

    def _fix_level1(self, item, valid_objects):
        item["interaction_objects"] = "none"
        item["num_interactions"] = 0
        if not valid_objects:
            item["instruction"] = re.sub(r"\b(left|right|front|back|center|centre|middle)\b", "area", item.get("instruction",""), flags=re.I)
            item["start_location"] = "Start"
            item["end_location"] = "Destination"
            return item
        obj_a, obj_b = self._pick_two_objects(valid_objects)
        item["start_location"] = f"Near {obj_a}"
        item["end_location"] = f"Near {obj_b}"
        item["instruction"] = f"Navigate from near the {obj_a} to near the {obj_b}."
        return item

    def _fix_locations_task(self, item, valid_objects, force_object_endpoints=True):
        instr = item.get("instruction", "")
        start = item.get("start_location", "")
        end = item.get("end_location", "")
        bad = self._contains_banned(instr) or self._contains_banned(start) or self._contains_banned(end)
        if force_object_endpoints and valid_objects and (bad or (not self._contains_any_object(start, valid_objects)) or (not self._contains_any_object(end, valid_objects))):
            obj_a, obj_b = self._pick_two_objects(valid_objects)
            item["start_location"] = f"Near {obj_a}"
            item["end_location"] = f"Near {obj_b}"
            if not self._contains_any_object(instr, valid_objects) or self._contains_banned(instr):
                item["instruction"] = f"Move from near the {obj_a} to near the {obj_b}."
        return item

    def _postprocess(self, parsed_by_level, valid_objects):
        valid_objects = [self._norm(v) for v in (valid_objects or []) if self._norm(v)]
        valid_objects = list(dict.fromkeys(valid_objects))
        for i in range(len(parsed_by_level.get("level_1_navigation", []))):
            parsed_by_level["level_1_navigation"][i] = self._fix_level1(parsed_by_level["level_1_navigation"][i], valid_objects)
        for key in ["level_2_simple_interaction", "level_3_complex_interaction"]:
            for it in parsed_by_level.get(key, []):
                it["interaction_objects"] = self._filter_interact_list(it.get("interaction_objects", ""), valid_objects)
                it["num_interactions"] = 0 if it["interaction_objects"] == "none" else len([x.strip() for x in it["interaction_objects"].split(",") if x.strip()])
                it = self._fix_locations_task(it, valid_objects, force_object_endpoints=True)
        return parsed_by_level

    def _parse_instructions_by_level(self, instructions_text):
        level_1, level_2, level_3 = [], [], []
        lines = instructions_text.strip().split('\n')
        for line in lines:
            line = line.strip()
            if not line or '|' not in line:
                continue
            try:
                if line.startswith('LEVEL_1'):
                    target = level_1; level = 1
                elif line.startswith('LEVEL_2'):
                    target = level_2; level = 2
                elif line.startswith('LEVEL_3'):
                    target = level_3; level = 3
                else:
                    continue
                parts = [p.strip() for p in line.split('|')]
                task, start, end, interact = "", "unspecified", "unspecified", "none"
                for part in parts:
                    if part.startswith('TASK:'):
                        task = part.replace('TASK:', '').strip()
                    elif part.startswith('START:'):
                        start = part.replace('START:', '').strip()
                    elif part.startswith('END:'):
                        end = part.replace('END:', '').strip()
                    elif part.startswith('INTERACT:'):
                        interact = part.replace('INTERACT:', '').strip()
                if task:
                    target.append({
                        "id": len(target) + 1,
                        "level": level,
                        "instruction": task,
                        "start_location": start,
                        "end_location": end,
                        "interaction_objects": interact,
                        "num_interactions": 0
                    })
            except Exception:
                continue
        return {
            "level_1_navigation": level_1,
            "level_2_simple_interaction": level_2,
            "level_3_complex_interaction": level_3
        }

    def generate_instructions_by_levels(self, scene_description, valid_objects, num_per_level=NUM_PER_LEVEL):
        valid_objects = valid_objects or []
        valid_objects_short = valid_objects[:25]
        valid_objects_str = ", ".join(valid_objects_short) if valid_objects_short else "unknown"
        banned_str = ", ".join(sorted(self.BANNED_LOCATION_TERMS))

        system_prompt = f"""You are a robot task instruction generator.

VALID_OBJECTS (use ONLY these object names):
{valid_objects_str}

HARD CONSTRAINTS:
- Never use viewpoint-dependent location phrases: {banned_str}.
- Do not use vague endpoints like "left side/right side/front/back/center".
- Use ONLY object landmarks: phrase endpoints as "Near <object>" where <object> is from VALID_OBJECTS.
- LEVEL_1: navigation-only; INTERACT must be "none"; must be object->object navigation.
- LEVEL_2/LEVEL_3: INTERACT must contain ONLY objects from VALID_OBJECTS (no locations).

Output exactly one task per line in this schema:
LEVEL_1 | TASK: ... | START: ... | END: ... | INTERACT: none
LEVEL_2 | TASK: ... | START: ... | END: ... | INTERACT: obj1, obj2
LEVEL_3 | TASK: ... | START: ... | END: ... | INTERACT: obj1, obj2, obj3
"""

        user_prompt = f"""Environment Description:
{scene_description}

Generate exactly {num_per_level} tasks for EACH level.

LEVEL 1 ({num_per_level}):
LEVEL_1 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: none

LEVEL 2 ({num_per_level}):
LEVEL_2 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: <object>, <object>

LEVEL 3 ({num_per_level}):
LEVEL_3 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: <object>, <object>, <object>

Generate all {num_per_level * 3} tasks now:"""

        completion = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=2048,
            temperature=0.6,
            top_p=0.9
        )
        instructions_text = completion.choices[0].message.content
        parsed_by_level = self._parse_instructions_by_level(instructions_text)
        parsed_by_level = self._postprocess(parsed_by_level, valid_objects)
        return parsed_by_level, instructions_text

In [12]:
def process_all_pairs(image_dir, output_root=OUTPUT_ROOT, num_per_level=NUM_PER_LEVEL):
    pairs = find_image_pairs(image_dir)
    if not pairs:
        print("No matching pairs found in", image_dir)
        return []

    print(f"Found {len(pairs)} pairs to process.")
    # instantiate models once
    scene_analyzer = SceneUnderstanding(model_name=LLAVA_MODEL_NAME, image_size=IMAGE_SIZE)
    instr_gen = InstructionGenerator(model_name=INSTRUCTION_LLM_MODEL, hf_api_key=HF_API_KEY)

    results = []
    for base, paths in sorted(pairs.items(), key=lambda x: x[0]):
        top_path = paths["2"]   # user said _2 is top-down
        front_path = paths["1"] # user said _1 is frontal
        print(f"\nProcessing base: '{base}'\n -> frontal: {front_path}\n -> topdown: {top_path}")

        out_dir = os.path.join(output_root, base.replace("/", "_"))
        os.makedirs(out_dir, exist_ok=True)

        try:
            # LLaVA scene understanding from two images
            analysis = scene_analyzer.analyze_two_views(topdown_path=top_path, frontal_path=front_path)
            scene_description_clean = analysis["clean_description"]
            valid_objects = analysis["valid_objects"]
            raw_llava = analysis["raw_output"]

            # save scene description
            sd_path = os.path.join(out_dir, f"{base}_scene_description.txt")
            with open(sd_path, "w") as f:
                f.write(scene_description_clean + "\n\nRAW_LLaVA_OUTPUT:\n" + raw_llava)

            # generate instructions with LLM
            parsed_by_level, raw_instructions = instr_gen.generate_instructions_by_levels(
                scene_description_clean, valid_objects, num_per_level=num_per_level
            )

            # Save outputs:
            # 1) raw instructions file named exactly "<base>.txt" as requested
            txt_path = os.path.join(output_root, f"{base}.txt")
            with open(txt_path, "w") as f:
                f.write(raw_instructions)

            # 2) structured JSON in the pair-specific out_dir
            json_path = os.path.join(out_dir, f"{base}_instructions.json")
            with open(json_path, "w") as f:
                json.dump(parsed_by_level, f, indent=2)

            # 3) also save a small summary file
            summary = {
                "base": base,
                "topdown": top_path,
                "frontal": front_path,
                "scene_description_file": sd_path,
                "instructions_txt": txt_path,
                "instructions_json": json_path,
                "objects": valid_objects
            }
            with open(os.path.join(out_dir, f"{base}_summary.json"), "w") as f:
                json.dump(summary, f, indent=2)

            print(f"✅ Done {base}: saved raw instructions -> {txt_path}")
            results.append({"base": base, "success": True, "out_dir": out_dir})
        except Exception as e:
            print(f"❌ Failed processing {base}: {e}")
            results.append({"base": base, "success": False, "error": str(e)})
        finally:
            torch.cuda.empty_cache()
            gc.collect()
    return results

In [13]:
pairs = find_image_pairs(IMAGE_DIR)
print("Pairs found (base -> files):")
for b, p in pairs.items():
    print(" ", b, "->", p.get("1"), p.get("2"))
results = process_all_pairs(IMAGE_DIR)
print("\nBatch finished. Results:")
for r in results:
    print(r)

Pairs found (base -> files):
  34_Pedestrian mall with.4 -> /content/gdrive/MyDrive/3DReasoningProject_images/Others/34_Pedestrian mall with.4_1.png /content/gdrive/MyDrive/3DReasoningProject_images/Others/34_Pedestrian mall with.4_2.png
  35_Bus loop with.3 -> /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.3_1.png /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.3_2.png
  35_Bus loop with.4 -> /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.4_1.png /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.4_2.png
  37_Fortress yard with.4 -> /content/gdrive/MyDrive/3DReasoningProject_images/Others/37_Fortress yard with.4_1.png /content/gdrive/MyDrive/3DReasoningProject_images/Others/37_Fortress yard with.4_2.png
Found 4 pairs to process.
Loading LLaVA model 'llava-hf/llava-1.5-7b-hf' on cuda ...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

✓ LLaVA ready

Processing base: '34_Pedestrian mall with.4'
 -> frontal: /content/gdrive/MyDrive/3DReasoningProject_images/Others/34_Pedestrian mall with.4_1.png
 -> topdown: /content/gdrive/MyDrive/3DReasoningProject_images/Others/34_Pedestrian mall with.4_2.png
✅ Done 34_Pedestrian mall with.4: saved raw instructions -> two_view_pipeline_results/34_Pedestrian mall with.4.txt

Processing base: '35_Bus loop with.3'
 -> frontal: /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.3_1.png
 -> topdown: /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.3_2.png
✅ Done 35_Bus loop with.3: saved raw instructions -> two_view_pipeline_results/35_Bus loop with.3.txt

Processing base: '35_Bus loop with.4'
 -> frontal: /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.4_1.png
 -> topdown: /content/gdrive/MyDrive/3DReasoningProject_images/Others/35_Bus loop with.4_2.png
✅ Done 35_Bus loop with.4: saved raw instructions -> two_vie